# 12. 특징 추출

Sobel·Scharr·Canny 에지와 Hough 직선·원 검출을 실습합니다.

> 예제 이미지는 노트북 옆 `data` 폴더에 넣으세요.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_image(name, flags=cv2.IMREAD_COLOR):
    path = Path("data") / name
    image = cv2.imread(str(path), flags)
    if image is None:
        raise FileNotFoundError(path)
    return image

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()


## Sobel·Scharr 그래디언트

In [ ]:
src = read_image("lenna.bmp")
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
scharr_x = cv2.Scharr(gray, cv2.CV_32F, 1, 0)
magnitude, angle = cv2.cartToPolar(sobel_x, sobel_y, angleInDegrees=True)
show([gray, cv2.convertScaleAbs(sobel_x), cv2.convertScaleAbs(sobel_y), cv2.convertScaleAbs(scharr_x), cv2.convertScaleAbs(magnitude)], ["gray", "Sobel x", "Sobel y", "Scharr x", "magnitude"])


## Canny 에지

In [ ]:
blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)
canny = cv2.Canny(blurred, 80, 160)
show([gray, blurred, canny], ["gray", "Gaussian blur", "Canny"])


## 확률적 Hough 직선 검출

In [ ]:
line_src = read_image("building.jpg")
line_gray = cv2.cvtColor(line_src, cv2.COLOR_BGR2GRAY)
line_edges = cv2.Canny(line_gray, 50, 150)
lines = cv2.HoughLinesP(line_edges, 1, np.pi / 180, threshold=70, minLineLength=50, maxLineGap=10)
line_result = line_src.copy()
if lines is not None:
    for x1, y1, x2, y2 in lines[:, 0]:
        cv2.line(line_result, (x1, y1), (x2, y2), (0, 0, 255), 2, cv2.LINE_AA)
show([line_src, line_edges, line_result], ["source", "edges", "Hough lines"])


## Hough 원 검출

In [ ]:
coin_src = read_image("coins.jpg")
coin_gray = cv2.cvtColor(coin_src, cv2.COLOR_BGR2GRAY)
coin_blur = cv2.GaussianBlur(coin_gray, (0, 0), 1.5)
circles = cv2.HoughCircles(coin_blur, cv2.HOUGH_GRADIENT, dp=1, minDist=30, param1=120, param2=30, minRadius=10, maxRadius=100)
coin_result = coin_src.copy()
circle_list = [] if circles is None else np.uint16(np.around(circles[0]))
for x, y, radius in circle_list:
    cv2.circle(coin_result, (x, y), radius, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.circle(coin_result, (x, y), 2, (255, 0, 0), 3, cv2.LINE_AA)
print("검출된 동전 수:", len(circle_list))
show([coin_src, coin_result], ["source", "detected circles"])


## 중심 HSV로 동전 종류 분류

In [ ]:
hsv = cv2.cvtColor(coin_src, cv2.COLOR_BGR2HSV)
coin_values = []
for x, y, radius in circle_list:
    hue = int(hsv[y, x, 0])
    value = 100 if 5 <= hue <= 25 else 500
    coin_values.append(value)
print("동전 값:", coin_values)
print("합계:", sum(coin_values), "원")
